In [ ]:
# Cell 1: imports & config

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
from scipy.stats import pearsonr
from sklearn.multioutput import MultiOutputClassifier
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported")

In [ ]:
# Cell 1: Check that files exist

# Configure paths
CONFIG = {
    "model_params_path": "results/MOC/LGBM_MOC_best_model_params.json",
    "threshold_path": "results/MOC/LGBM_MOC_best_thresholds.json",
    "data_paths": {
        "X_train": "data/X_train.csv", 
        "y_train": "data/Y_train.csv"
    }
}

def check_files_exist():
    """Check that all required files exist"""
    print("=" * 60)
    print("File existence check")
    print("=" * 60)
    
    files_to_check = [
        ("Model parameters", CONFIG['model_params_path']),
        ("Threshold file", CONFIG['threshold_path']),
        ("Feature data", CONFIG['data_paths']['X_train']),
        ("Target variables", CONFIG['data_paths']['y_train'])
    ]
    
    all_exist = True
    for file_desc, file_path in files_to_check:
        if os.path.exists(file_path):
            print(f"✓ {file_desc}: {file_path} - exists")
            # Show file size
            file_size = os.path.getsize(file_path)
            print(f"  File size: {file_size} bytes")
        else:
            print(f"✗ {file_desc}: {file_path} - missing")
            all_exist = False
    
    return all_exist

# Check file existence
files_exist = check_files_exist()

In [ ]:
# Cell 2: Configuration and data loading
# Configure paths
CONFIG = {
    "model_params_path": "results/MOC/LGBM_MOC_best_model_params.json",
    "threshold_path": "results/MOC/LGBM_MOC_best_thresholds.json",
    "data_paths": {
        "X_train": "data/X_train.csv", 
        "y_train": "data/Y_train.csv"
    },
    "output": {
        "residuals": "results/residuals/LGBM_MOC_train_residuals.csv",
        "binary_residuals": "results/residuals/LGBM_MOC_train_binary_residuals.csv",
        "partial_correlation": "results/correlation/incident_type_partial_correlations.csv",
        "correlation_plot": "results/correlation/partial_correlation_heatmap.png"
    }
}

def clean_feature_names_strict(df):
    """Strictly clean feature names, keeping only letters, digits and underscores"""
    cleaned_df = df.copy()
    new_columns = []
    for col in cleaned_df.columns:
        # Keep only letters, digits and underscores; replace other characters with underscores
        col_clean = re.sub(r'[^a-zA-Z0-9_]', '_', str(col))
        # Collapse repeated underscores
        col_clean = re.sub(r'_+', '_', col_clean)
        # Strip leading and trailing underscores
        col_clean = col_clean.strip('_')
        # Use a default name if empty
        if not col_clean:
            col_clean = 'feature'
        new_columns.append(col_clean)
    
    # Ensure column names are unique
    unique_columns = []
    seen = {}
    for col in new_columns:
        if col in seen:
            seen[col] += 1
            unique_columns.append(f"{col}_{seen[col]}")
        else:
            seen[col] = 0
            unique_columns.append(col)
    
    cleaned_df.columns = unique_columns
    return cleaned_df

# Load data
def load_data():
    """Load training data and strictly clean feature names"""
    X_train = pd.read_csv(CONFIG['data_paths']['X_train'])
    y_train = pd.read_csv(CONFIG['data_paths']['y_train'])
    
    print("Example raw feature names:")
    print(list(X_train.columns)[:10])
    
    # Strictly clean feature names
    X_train = clean_feature_names_strict(X_train)
    
    print(f"Training data shape: X_train {X_train.shape}, y_train {y_train.shape}")
    print(f"Targets: {list(y_train.columns)}")
    print(f"Example cleaned feature names:")
    print(list(X_train.columns)[:10])
    
    return X_train, y_train

X_train, y_train = load_data()

In [ ]:
# Cell 3: Load model parameters and thresholds, initialize model
def load_model_params_and_thresholds(y_train_columns):
    """Load model parameters and thresholds"""
    with open(CONFIG['model_params_path'], 'r') as f:
        model_params = json.load(f)
    
    # Load thresholds
    try:
        with open(CONFIG['threshold_path'], 'r') as f:
            thresholds_data = json.load(f)
        
        # Handle threshold data (list or dict)
        if isinstance(thresholds_data, list):
            # If a list, assume the order matches y_train.columns
            thresholds = {col: threshold for col, threshold in zip(y_train_columns, thresholds_data)}
            print("Thresholds are a list; converted to dict")
        elif isinstance(thresholds_data, dict):
            thresholds = thresholds_data
            print("Thresholds are a dict")
        else:
            print("Unknown threshold format; using default 0.5")
            thresholds = {col: 0.5 for col in y_train_columns}
            
        print("Threshold file loaded")
    except FileNotFoundError:
        print("Threshold file not found; using default 0.5")
        thresholds = {col: 0.5 for col in y_train_columns}
    
    return model_params, thresholds

def initialize_model(params):
    """Initialize multi-output classifier"""
    base_model = LGBMClassifier(
        verbosity=-1,
        random_state=42
    )
    base_model.set_params(**params)
    return MultiOutputClassifier(base_model)

# Reload model parameters and thresholds
model_params, thresholds = load_model_params_and_thresholds(y_train.columns)
model = initialize_model(model_params)

print("Model initialized")
print(f"Thresholds used: {thresholds}")

In [ ]:
# Cell 4: Train model
print("Training model...")
model.fit(X_train, y_train)
print("Model training finished!")

In [ ]:
# Cell 5: Compute training-set residuals (continuous-probability and binary)
def calculate_residuals(model, X_train, y_train, thresholds):
    """Compute residuals on the training set"""
    # Get predicted probabilities
    y_pred_proba = model.predict_proba(X_train)
    
    # Create DataFrames for continuous and binary residuals
    continuous_residuals = pd.DataFrame()
    binary_residuals = pd.DataFrame()
    binary_predictions = pd.DataFrame()
    
    for i, target_name in enumerate(y_train.columns):
        # Get predicted probability for target i
        if hasattr(y_pred_proba[i], 'shape') and y_pred_proba[i].shape[1] == 2:
            pred_proba = y_pred_proba[i][:, 1]
        else:
            pred_proba = y_pred_proba[i]
        
        # Get the threshold for this target
        threshold = thresholds.get(target_name, 0.5)
        
        # Continuous residual: actual - predicted probability
        continuous_residuals[target_name] = y_train[target_name] - pred_proba
        
        # Binary prediction: apply threshold
        binary_pred = (pred_proba >= threshold).astype(int)
        binary_predictions[target_name] = binary_pred
        
        # Binary residual: actual - binary prediction
        binary_residuals[target_name] = y_train[target_name] - binary_pred
    
    return continuous_residuals, binary_residuals, binary_predictions

continuous_residuals, binary_residuals, binary_predictions = calculate_residuals(model, X_train, y_train, thresholds)

print("Residual computation finished")
print("\nContinuous residual summary:")
print(continuous_residuals.describe())
print("\nBinary residual summary:")
print(binary_residuals.describe())

# Save residuals
continuous_residuals.to_csv(CONFIG['output']['residuals'], index=False)
binary_residuals.to_csv(CONFIG['output']['binary_residuals'], index=False)
print(f"\nContinuous residuals saved to: {CONFIG['output']['residuals']}")
print(f"Binary residuals saved to: {CONFIG['output']['binary_residuals']}")

In [ ]:
# Cell 7: Compute partial correlations (from binary residuals)
def calculate_partial_correlations(binary_residuals):
    """
    Compute partial correlations between incident types from binary residuals
    Correlation of binary residuals reflects the conditional dependence between prediction errors of different incident types, given the features and thresholds
    """
    # Compute the correlation matrix of binary residuals
    partial_corr_matrix = binary_residuals.corr()
    
    # Extract partial correlations for all variable pairs
    partial_corr_pairs = []
    incident_types = binary_residuals.columns
    
    for i in range(len(incident_types)):
        for j in range(i+1, len(incident_types)):
            type1 = incident_types[i]
            type2 = incident_types[j]
            corr_value = partial_corr_matrix.iloc[i, j]
            
            # Compute p-value
            corr, p_value = pearsonr(binary_residuals[type1], binary_residuals[type2])
            
            partial_corr_pairs.append({
                'incident_type_1': type1,
                'incident_type_2': type2,
                'partial_correlation': corr_value,
                'abs_partial_correlation': abs(corr_value),
                'p_value': p_value,
                'significant': p_value < 0.05
            })
    
    # Convert to DataFrame and sort
    partial_corr_df = pd.DataFrame(partial_corr_pairs)
    partial_corr_df = partial_corr_df.sort_values('abs_partial_correlation', ascending=False)
    
    return partial_corr_matrix, partial_corr_df

partial_corr_matrix, partial_corr_df = calculate_partial_correlations(binary_residuals)

print("Partial correlation computation finished")
print(f"Partial correlation matrix shape: {partial_corr_matrix.shape}")

In [ ]:
# Cell 9: Conditional independence analysis
def analyze_conditional_independence(partial_corr_df, independence_threshold=0.1):
    """
    Analyze conditional independence
    Partial correlations of binary residuals near 0 indicate that prediction errors of incident types are conditionally independent given the features and thresholds
    """
    print("=" * 70)
    print("Conditional independence results (binary residuals)")
    print("=" * 70)
    
    total_pairs = len(partial_corr_df)
    significant_pairs = partial_corr_df[partial_corr_df['significant']]
    
    # Count pairs at each correlation level
    weak_corr = partial_corr_df[partial_corr_df['abs_partial_correlation'] < independence_threshold]
    moderate_corr = partial_corr_df[(partial_corr_df['abs_partial_correlation'] >= independence_threshold) & 
                                   (partial_corr_df['abs_partial_correlation'] < 0.3)]
    strong_corr = partial_corr_df[partial_corr_df['abs_partial_correlation'] >= 0.3]
    
    print(f"Total incident-type pairs: {total_pairs}")
    print(f"Significantly correlated pairs (p < 0.05): {len(significant_pairs)}")
    
    print(f"\nCorrelation distribution:")
    print(f"Weak (|ρ| < {independence_threshold}): {len(weak_corr)} pairs ({len(weak_corr)/total_pairs*100:.1f}%)")
    print(f"Moderate (0.1 ≤ |ρ| < 0.3): {len(moderate_corr)} pairs ({len(moderate_corr)/total_pairs*100:.1f}%)")
    print(f"Strong (|ρ| ≥ 0.3): {len(strong_corr)} pairs ({len(strong_corr)/total_pairs*100:.1f}%)")
    
    print(f"\nOverall statistics:")
    print(f"Mean absolute partial correlation: {partial_corr_df['abs_partial_correlation'].mean():.4f}")
    print(f"Median partial correlation: {partial_corr_df['partial_correlation'].abs().median():.4f}")
    print(f"Std of partial correlation: {partial_corr_df['partial_correlation'].std():.4f}")
    
    # Show the strongest correlated pairs
    if len(strong_corr) > 0:
        print(f"\nTop 5 strongest conditional dependencies:")
        for _, row in strong_corr.head().iterrows():
            sig_flag = "**" if row['significant'] else ""
            print(f"  {row['incident_type_1']} - {row['incident_type_2']}: {row['partial_correlation']:.3f}{sig_flag}")
    
    # Conditional independence conclusion
    weak_percentage = len(weak_corr) / total_pairs * 100
    print(f"\nConditional independence conclusion:")
    if weak_percentage > 90:
        print("✓ Strong evidence for conditional independence: over 90% of incident-type pairs are conditionally independent given the features and thresholds")
    elif weak_percentage > 80:
        print("○ Moderate evidence for conditional independence: 80-90% of incident-type pairs are conditionally independent given the features and thresholds") 
    elif weak_percentage > 70:
        print("△ Weak evidence for conditional independence: 70-80% of incident-type pairs are conditionally independent given the features and thresholds")
    else:
        print("✗ No evidence for conditional independence: fewer than 70% of incident-type pairs are conditionally independent given the features and thresholds")
    
    return weak_corr, moderate_corr, strong_corr

weak_corr, moderate_corr, strong_corr = analyze_conditional_independence(partial_corr_df)